<a href="https://colab.research.google.com/github/Akashkumar12/AI-ML/blob/main/U2_MH1_AuthorIdentification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>




# Advanced Certification in AIML
## A Program by IIIT-H and TalentSprint

## Problem Statement

The problem is to identify the author of a  book from a given list of possible authors.

## Learning Objectives

At the end of the experiment, you will be able to:

* Use NLTK package
* Extract handcrafted features
* Preprocess the text
* Write an algorithm to identify the author of a given book


In [ ]:
#@title  Mini Hackathon Walkthrough
from IPython.display import HTML

HTML("""<video width="854" height="480" controls>
  <source src="https://cdn.iiith.talentsprint.com/aiml/Experiment_related_data/Walkthrough/authoridentification.mp4" type="video/mp4">
</video>
""")

## Background

Author identification is the task of identifying the author of a given text. It can be considered as a typical classification problem, where a set of books with known authors are used for training. The aim is to automatically determine the corresponding author of an anonymous text.

## Grading = 10 Marks

## Setup Steps

In [1]:
#@title Run this cell to complete the setup for this Notebook

from IPython import get_ipython
ipython = get_ipython()

notebook="U2_MH1_AuthorIdentification" #name of the notebook
Answer = "This notebook is graded by mentors on the day of hackathon"
def setup():
    ipython.magic("sx wget https://cdn.talentsprint.com/talentsprint1/archives/sc/aiml/experiment_related_data/AIML_DS_GOOGLENEWS-VECTORS-NEGATIVE-300_STD.rar")
    ipython.magic("sx unrar e /content/AIML_DS_GOOGLENEWS-VECTORS-NEGATIVE-300_STD.rar")
    print ("Setup completed successfully")
    return

setup()

Setup completed successfully


### NOTE: You are allowed to use ML libraries such as Sklearn, NLTK, etc wherever applicable

### Downloading the required nltk Packages before moving ahead

In [2]:
import nltk
nltk.download('gutenberg')
nltk.download('punkt')

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

## **Stage 1:** Dataset Preparation

### 1 Marks -> Ensure you appropriately split the multiple short stories for the below-mentioned authors, Which will be your training data.

**1.** Before moving ahead choose two authors based on your team-number allocation: <br/>


Team=1,5,9,13,17,21  &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;    Author-A Vs Author-B <br />
Team=2,6,10,14,18,22 &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;         Author-B Vs Author-C <br />
Team=3,7,11,15,19,23 &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;         Author-C Vs Author-D <br />
Team=4,8,12,16,20,24 &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;           Author-D Vs Author-E <br />



**2.** Link to the short stories collection of each author for your problem: <br />

*   Author-A -> Rudyard Kipling   [Short Stories Collection](http://www.gutenberg.org/files/2781/2781-0.txt) &nbsp;&nbsp;
*   Author-B -> Anton Chekhov [Short Stories Collection](http://www.gutenberg.org/files/1732/1732-0.txt) &nbsp;&nbsp;
*   Author-C -> Guy De Maupassant [Short Stories Collection](http://www.gutenberg.org/cache/epub/21327/pg21327.txt)&nbsp;&nbsp;
*   Author-D -> Mark Twain [Short Stories Collection](http://www.gutenberg.org/files/245/245-0.txt)&nbsp;&nbsp;
*   Author-E -> Saki [Short Stories Collection](http://www.gutenberg.org/files/1477/1477-0.txt)&nbsp;&nbsp;

**Hint for downloading raw text from Gutenberg :**  Refer to the section "Electronic Books" in the following  [link](https://www.nltk.org/book/ch03.html) for the instructions.



**Hint for finding the index of a text:**   You may use `raw.find()` and `raw.rfind()` in the same [link](https://www.nltk.org/book/ch03.html) to find the appropriate index of the start and end location

**Hint for splitting the multiple stories:** Split the stories using long space (white space character)

**Note:** Ignore the table of contents section from the given stories

In [3]:
# YOUR CODE HERE for downloading and splitting the multiple stories of respective authors which are allocated to you
import re
import pandas as pd
from urllib.request import Request, urlopen
from sklearn.model_selection import train_test_split

In [4]:
authors = {
    "Anton Chekhov": {
        "url": "https://www.gutenberg.org/files/1732/1732-0.txt",
        "first_story": "THE SCHOOLMISTRESS"
    },
    "Guy de Maupassant": {
        "url": "https://www.gutenberg.org/cache/epub/21327/pg21327.txt",
        "first_story": "BOULE DE SUIF"
    }
}

In [5]:
def download_text(url):
    request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(request) as response:
        return response.read().decode("utf-8", errors="ignore")


def remove_gutenberg_parts(raw):
    raw = raw.replace("\r\n", "\n").replace("\r", "\n")
    start_match = re.search(r"\*\*\* START OF .*? \*\*\*", raw, re.I | re.S)
    if start_match:
        raw = raw[start_match.end():]

    end_index = raw.upper().rfind("*** END OF")
    if end_index != -1:
        raw = raw[:end_index]

    return raw.strip()


def skip_table_of_contents(text, first_story):
    upper_text = text.upper()
    first_story = first_story.upper()

    first_index = upper_text.find(first_story)
    second_index = upper_text.find(first_story, first_index + len(first_story))

    if second_index != -1:
        return text[second_index:].strip()
    if first_index != -1:
        return text[first_index:].strip()

    return text.strip()


def split_stories(text, min_words=300):
    # Long whitespace gaps separate stories in these Gutenberg collections.
    parts = re.split(r"\n\s*\n\s*\n\s*\n+", text)
    stories = []

    for part in parts:
        part = part.strip()
        if len(part.split()) >= min_words:
            stories.append(part)

    return stories

In [6]:
records = []

for author, info in authors.items():
    raw = download_text(info["url"])
    clean_text = remove_gutenberg_parts(raw)
    story_text = skip_table_of_contents(clean_text, info["first_story"])
    stories = split_stories(story_text)

    print(author, "stories found:", len(stories))

    for story_id, story in enumerate(stories, start=1):
        records.append({
            "author": author,
            "story_id": story_id,
            "text": story
        })

df = pd.DataFrame(records)
df.head()

Anton Chekhov stories found: 21
Guy de Maupassant stories found: 30


,author,story_id,text
0,Anton Chekhov,1,THE SCHOOLMISTRESS\n\nAT half-past eight they ...
1,Anton Chekhov,2,A NERVOUS BREAKDOWN\n\nA MEDICAL student calle...
2,Anton Chekhov,3,MISERY\n\n“To whom shall I tell my grief?”\n\n...
3,Anton Chekhov,4,CHAMPAGNE\n\nA WAYFARER’S STORY\n\nIN the year...
4,Anton Chekhov,5,AFTER THE THEATRE\n\nNADYA ZELENIN had just co...


In [32]:
print(df["author"].value_counts())
df.to_csv("group2_author_dataset.csv", index=False)
print("Dataset saved as group2_author_dataset.csv")

author
Guy de Maupassant    30
Anton Chekhov        21
Name: count, dtype: int64
Dataset saved as group2_author_dataset.csv


## **Stage 2**: Experiment with Handcrafted features representation
Extract Handcrafted features for the obtained short stories from **Stage-1**

**Stylometry:**

Each person has a unique vocabulary, sometimes rich, sometimes limited. Although a larger vocabulary is usually associated with literary quality, this is not always the case. Ernest Hemingway is famous for using a surprisingly small number of different words in his writing, which did not prevent him from winning the Nobel Prize for Literature in 1954.

Some people write in short sentences, while others prefer long blocks of text consisting of many clauses. No two people use semicolons, em-dashes, and other forms of punctuation in the same way.




**You may explore the following ways to analyze the text and generate handcrafted features by searching text in a probing way:**

a)  Could the style of punctuation usage help as a handcrafted feature? Both by those who follow punctuations and by those who don't? Interesting [link](https://qwiklit.com/2014/03/05/top-10-authors-who-ignored-the-basic-rules-of-punctuation/)

b) The same word can sometimes be used in different contexts repeatedly by different authors. Could this fact be converted as a handcrafted feature? [link](https://www.nltk.org/book/ch01.html)

c) The above two are merely examples; As you might have noticed already the NLTK book [link](https://www.nltk.org/book/) offers several methods of analyzing and understanding the text. Each of these analyses is in itself capable of being a handcrafted feature. **However for your evaluation a minimal set of useful handcrafted features which is helping you prove a classification of an is sufficient**

d) Could most common words be used to distinguish authors?  Refer "Counting Vocabulary" section of the [link](https://www.nltk.org/book/ch01.html)

e) How about using a count of most frequently used bi-gram, tri-grams, and using it to classify an author?

f) How about using the frequency histogram of the most frequently used words across the stories by a given author a useful feature?

The limit here is endlessly limited only by your imagination, and of course your accuracy! :)


### 1 Marks ->  a) List 6 handcrafted features to distinguish author stories.

In [9]:
import string
from collections import Counter
def get_words(text):
    return re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text.lower())

def get_sentences(text):
    sentences = re.split(r"[.!?]+", text)
    return [sentence.strip() for sentence in sentences if sentence.strip()]

def extract_handcrafted_features(text):
    words = get_words(text)
    sentences = get_sentences(text)
    punctuation_count = sum(1 for char in text if char in string.punctuation)

    total_words = len(words)
    unique_words = len(set(words))
    word_counts = Counter(words)
    most_common_count = word_counts.most_common(1)[0][1] if total_words > 0 else 0

    return {
        "UniqueWords": unique_words,
        "AvgSentLength": total_words / len(sentences) if sentences else 0,
        "TypeTokenRatio": unique_words / total_words if total_words else 0,
        "AvgWordLength": sum(len(word) for word in words) / total_words if total_words else 0,
        "PunctuationCount": punctuation_count,
        "MostCommonWordFreq": most_common_count / total_words if total_words else 0
    }

In [10]:
# For eg:
# 1. UniqueWords
# 2. AvgSentLength
# List the other handcrafted features here
# Use df from Stage 1. If needed, uncomment the next line.
# df = pd.read_csv("group2_author_dataset.csv")

feature_rows = []

for _, row in df.iterrows():
    features = extract_handcrafted_features(row["text"])
    features["author"] = row["author"]
    features["story_id"] = row["story_id"]
    feature_rows.append(features)

features_df = pd.DataFrame(feature_rows)

features_df = features_df[
    [
        "author",
        "story_id",
        "UniqueWords",
        "AvgSentLength",
        "TypeTokenRatio",
        "AvgWordLength",
        "PunctuationCount",
        "MostCommonWordFreq"
    ]
]

features_df.head()

,author,story_id,UniqueWords,AvgSentLength,TypeTokenRatio,AvgWordLength,PunctuationCount,MostCommonWordFreq
0,Anton Chekhov,1,975,16.764423,0.279610,4.285346,611,0.069974
1,Anton Chekhov,2,1836,14.922428,0.207481,4.286247,1692,0.057973
2,Anton Chekhov,3,690,9.653153,0.321979,3.976202,581,0.050863
3,Anton Chekhov,4,775,17.492308,0.340809,4.112577,325,0.051891
4,Anton Chekhov,5,418,20.517857,0.363795,4.199304,179,0.054830


In [11]:
features_df.to_csv("group2_handcrafted_features.csv", index=False)

print("Feature rows:", len(features_df))
print("Saved as group2_handcrafted_features.csv")
features_df.groupby("author").mean(numeric_only=True)

Feature rows: 51
Saved as group2_handcrafted_features.csv


,story_id,UniqueWords,AvgSentLength,TypeTokenRatio,AvgWordLength,PunctuationCount,MostCommonWordFreq
author,,,,,,,
Anton Chekhov,11.0,859.000000,15.885306,0.310143,4.174985,573.380952,0.064490
Guy de Maupassant,15.5,958.433333,18.999289,0.301332,4.147765,759.766667,0.052189


###  2 Marks -> b) Write functions for any 4 of the above 6 handcrafted features and label your authors accordingly.

- Get any 4 hand crafted features from the above listed 6 hand-crafted features for every story obtained from **stage-1**.
- Identify your target variable as an author and label them accordingly.

In [33]:
import string

AUTHOR_LABELS = {
    "Anton Chekhov": 0,
    "Guy de Maupassant": 1
}

def feature_unique_words(text):
    words = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text.lower())
    return len(set(words))

def feature_avg_sentence_length(text):
    words = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text.lower())
    sentences = re.split(r"[.!?]+", text)
    sentences = [sentence.strip() for sentence in sentences if sentence.strip()]

    if len(sentences) == 0:
        return 0

    return len(words) / len(sentences)

def feature_type_token_ratio(text):
    words = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text.lower())

    if len(words) == 0:
        return 0

    return len(set(words)) / len(words)

def feature_punctuation_count(text):
    return sum(1 for char in text if char in string.punctuation)

In [13]:
# Stories_list    UniqueWords    AvgSentLength     Label
#     1               x1               x2            y

# YOUR CODE HERE
# Use df from Stage 1. If needed, uncomment the next line.
# df = pd.read_csv("group2_author_dataset.csv")

four_feature_rows = []

for _, row in df.iterrows():
    text = row["text"]
    author = row["author"]

    four_feature_rows.append({
        "author": author,
        "label": AUTHOR_LABELS[author],
        "story_id": row["story_id"],
        "UniqueWords": feature_unique_words(text),
        "AvgSentLength": feature_avg_sentence_length(text),
        "TypeTokenRatio": feature_type_token_ratio(text),
        "PunctuationCount": feature_punctuation_count(text)
    })

stage2b_df = pd.DataFrame(four_feature_rows)
stage2b_df.to_csv("group2_four_handcrafted_features.csv", index=False)

print("Author label mapping:", AUTHOR_LABELS)
print("Saved as group2_four_handcrafted_features.csv")
stage2b_df.head()

Author label mapping: {'Anton Chekhov': 0, 'Guy de Maupassant': 1}
Saved as group2_four_handcrafted_features.csv


,author,label,story_id,UniqueWords,AvgSentLength,TypeTokenRatio,PunctuationCount
0,Anton Chekhov,0,1,975,16.764423,0.279610,611
1,Anton Chekhov,0,2,1836,14.922428,0.207481,1692
2,Anton Chekhov,0,3,690,9.653153,0.321979,581
3,Anton Chekhov,0,4,775,17.492308,0.340809,325
4,Anton Chekhov,0,5,418,20.517857,0.363795,179


##**Stage 3:** Experiment with Text processing and representation:
Extract features using TFIDF or CountVectorizer or Word2vec for the obtained short stories from **Stage-1**



### 1 Mark -> a) Performing basic cleanup operations such as removing the newline characters and removing trailing spaces

**For example,** Your sentence looks as follows \[' This is a sentence\n\r. Another sentence \n'].

After newline removal from the above example, your sentence will look like \['This is a sentence. Another sentence'].

 In order to do this, you can try using a combination of split() and join()

In [14]:
# YOUR CODE HERE
def basic_cleanup(text):
    # split() removes newlines, tabs, and extra spaces; join() rebuilds clean text.
    return " ".join(text.split())

In [15]:
# Example from the question
sample_sentence = " This is a sentence\n\r. Another sentence \n"

print("Before cleanup:")
print([sample_sentence])

print("After cleanup:")
print([basic_cleanup(sample_sentence)])

Before cleanup:
[' This is a sentence\n\r. Another sentence \n']
After cleanup:
['This is a sentence . Another sentence']


In [16]:
# Use df from Stage 1. If needed, uncomment the next line.
# df = pd.read_csv("group2_author_dataset.csv")

cleaned_df = df.copy()
cleaned_df["clean_text"] = cleaned_df["text"].apply(basic_cleanup)

cleaned_df[["author", "story_id", "clean_text"]].head()

,author,story_id,clean_text
0,Anton Chekhov,1,THE SCHOOLMISTRESS AT half-past eight they dro...
1,Anton Chekhov,2,A NERVOUS BREAKDOWN A MEDICAL student called M...
2,Anton Chekhov,3,MISERY “To whom shall I tell my grief?” THE tw...
3,Anton Chekhov,4,CHAMPAGNE A WAYFARER’S STORY IN the year in wh...
4,Anton Chekhov,5,AFTER THE THEATRE NADYA ZELENIN had just come ...


In [17]:
cleaned_df.to_csv("group2_cleaned_stories.csv", index=False)

print("Cleaned rows:", len(cleaned_df))
print("Saved as group2_cleaned_stories.csv")

Cleaned rows: 51
Saved as group2_cleaned_stories.csv


###  2 Marks-> b) Generate vectors for the given stories

Create a representation of text, convert it into vectors (numbers)


**Use any one** of the following algorithms for this task :

* Countvectorizer or
* TFIDFVectorizer or
* Word2Vec (The word2vec bin file (AIML_DS_GOOGLENEWS-VECTORS-NEGATIVE-300_STD) can be downloaded as a part of setup  )
  * perform sentence level tokenization and word level tokenization for the given stories

    **Example of sentences as list of words:**<br/>
    **Before:** ['This is a sentence .' , ' Another sentence']<br/>
    **After:** ['This', 'is' ,'a', 'sentence' , ' . ' , ' Another ', ' sentence ' ]
 * Assign the respective label associated for each vector representation of the extracted word

References Documents:

1.   [Countvectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)
2.  [TFIDFVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)


In [18]:
# YOUR CODE HERE (HINT: Convert to numpy array if needed)
from sklearn.feature_extraction.text import TfidfVectorizer
# Use cleaned_df from Stage 3(a). If needed, uncomment the next line.
# cleaned_df = pd.read_csv("group2_cleaned_stories.csv")

AUTHOR_LABELS = {
    "Anton Chekhov": 0,
    "Guy de Maupassant": 1
}

cleaned_df["label"] = cleaned_df["author"].map(AUTHOR_LABELS)

tfidf_vectorizer = TfidfVectorizer(
    max_features=1000,
    lowercase=True,
    stop_words="english"
)

X_tfidf = tfidf_vectorizer.fit_transform(cleaned_df["clean_text"])
y = cleaned_df["label"]

print("TF-IDF vector shape:", X_tfidf.shape)
print("Labels shape:", y.shape)
print("Author label mapping:", AUTHOR_LABELS)

TF-IDF vector shape: (51, 1000)
Labels shape: (51,)
Author label mapping: {'Anton Chekhov': 0, 'Guy de Maupassant': 1}


In [19]:
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=tfidf_feature_names
)

tfidf_df.insert(0, "label", cleaned_df["label"].values)
tfidf_df.insert(0, "story_id", cleaned_df["story_id"].values)
tfidf_df.insert(0, "author", cleaned_df["author"].values)

tfidf_df.to_csv("group2_tfidf_vectors.csv", index=False)

print("Saved as group2_tfidf_vectors.csv")
tfidf_df.head()

Saved as group2_tfidf_vectors.csv


,author,story_id,label,_august,abbe,able,acquaintance,act,actress,adventure,...,writing,written,yard,yasha,year,years,yellow,yes,yesterday,young
0,Anton Chekhov,1,0,0.0,0.0,0.0,0.000000,0.00000,0.0,0.0,...,0.014687,0.000000,0.000000,0.0,0.023709,0.062413,0.000000,0.000000,0.000000,0.039337
1,Anton Chekhov,2,0,0.0,0.0,0.0,0.005171,0.00606,0.0,0.0,...,0.006060,0.010648,0.000000,0.0,0.014673,0.007357,0.005171,0.019476,0.004891,0.042199
2,Anton Chekhov,3,0,0.0,0.0,0.0,0.000000,0.00000,0.0,0.0,...,0.000000,0.000000,0.049423,0.0,0.000000,0.000000,0.000000,0.017014,0.012818,0.042536
3,Anton Chekhov,4,0,0.0,0.0,0.0,0.028037,0.00000,0.0,0.0,...,0.000000,0.000000,0.034086,0.0,0.132608,0.000000,0.000000,0.017601,0.000000,0.088007
4,Anton Chekhov,5,0,0.0,0.0,0.0,0.045109,0.00000,0.0,0.0,...,0.105734,0.046445,0.054840,0.0,0.000000,0.000000,0.000000,0.000000,0.042670,0.000000


###  1 Mark -> c) Is stop word removal necessary in the context of author identification? Your thoughts below?

In [ ]:
# YOUR ANSWER IN TEXT

# YOUR ANSWER IN TEXT
Stop word removal is not always necessary for author identification. In normal topic classification,
stop words are often removed because words like "the", "is", "and", and "of" may not carry strong topic meaning.
However, in author identification, these common words can be useful because different authors may use function words in different patterns.
An author writing style is often reflected in small word choices, sentence flow, and repeated use of common words.
Removing stop words may remove useful stylometric information. Therefore, for author identification,
it is better to experiment with both versions: one with stop word removal and one without stop word removal, then compare the classification accuracy.

##**Stage 4:** Classification :

### Expected accuracy is above 85%

### 2 Marks -> Perform a classification using either features obtained from Stage2 or Stage3

In [26]:
# YOUR CODE HERE
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [27]:
# Use cleaned_df from Stage 3(a). If needed, uncomment the next line.
# cleaned_df = pd.read_csv("group2_cleaned_stories.csv")

AUTHOR_LABELS = {
    "Anton Chekhov": 0,
    "Guy de Maupassant": 1
}

model_df = cleaned_df.copy()
model_df["label"] = model_df["author"].map(AUTHOR_LABELS)

X = model_df["clean_text"]
y = model_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

improved_classifier = Pipeline([
    ("features", FeatureUnion([
        ("word_tfidf", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 3),
            lowercase=True,
            sublinear_tf=True,
            max_features=20000
        )),
        ("char_tfidf", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            lowercase=True,
            sublinear_tf=True,
            max_features=30000
        ))
    ])),
    ("model", LinearSVC(C=1.0, random_state=42))
])

improved_classifier.fit(X_train, y_train)

Pipeline(steps=[('features',
                 FeatureUnion(transformer_list=[('word_tfidf',
                                                 TfidfVectorizer(max_features=20000,
                                                                 ngram_range=(1,
                                                                              3),
                                                                 sublinear_tf=True)),
                                                ('char_tfidf',
                                                 TfidfVectorizer(analyzer='char_wb',
                                                                 max_features=30000,
                                                                 ngram_range=(3,
                                                                              5),
                                                                 sublinear_tf=True))])),
                ('model', LinearSVC(random_state=42))])

In [28]:
y_pred = improved_classifier.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Improved Accuracy:", round(accuracy * 100, 2), "%")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Anton Chekhov", "Guy de Maupassant"]
))

Improved Accuracy: 100.0 %

Confusion Matrix:
[[5 0]
 [0 6]]

Classification Report:
                   precision    recall  f1-score   support

    Anton Chekhov       1.00      1.00      1.00         5
Guy de Maupassant       1.00      1.00      1.00         6

         accuracy                           1.00        11
        macro avg       1.00      1.00      1.00        11
     weighted avg       1.00      1.00      1.00        11



Optional: Chunk-Based Classification

In [29]:
def make_chunks(text, chunk_size=500, overlap=100, min_words=200):
    words = text.split()

    if len(words) <= chunk_size:
        return [" ".join(words)]

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk_words = words[start:start + chunk_size]

        if len(chunk_words) >= min_words:
            chunks.append(" ".join(chunk_words))

    return chunks


def create_chunk_dataset(dataframe):
    rows = []

    for source_index, row in dataframe.iterrows():
        chunks = make_chunks(row["clean_text"])

        for chunk_id, chunk_text in enumerate(chunks):
            rows.append({
                "source_index": source_index,
                "story_id": row["story_id"],
                "author": row["author"],
                "label": row["label"],
                "chunk_id": chunk_id,
                "chunk_text": chunk_text
            })

    return pd.DataFrame(rows)

In [30]:
train_story_df, test_story_df = train_test_split(
    model_df,
    test_size=0.2,
    random_state=42,
    stratify=model_df["label"]
)

train_chunks_df = create_chunk_dataset(train_story_df)
test_chunks_df = create_chunk_dataset(test_story_df)

print("Training chunks:", len(train_chunks_df))
print("Testing chunks:", len(test_chunks_df))

Training chunks: 317
Testing chunks: 109


In [31]:
chunk_classifier = Pipeline([
    ("features", FeatureUnion([
        ("word_tfidf", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 3),
            lowercase=True,
            sublinear_tf=True,
            max_features=20000
        )),
        ("char_tfidf", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            lowercase=True,
            sublinear_tf=True,
            max_features=30000
        ))
    ])),
    ("model", LinearSVC(C=1.0, random_state=42))
])

chunk_classifier.fit(train_chunks_df["chunk_text"], train_chunks_df["label"])

chunk_predictions = chunk_classifier.predict(test_chunks_df["chunk_text"])
chunk_accuracy = accuracy_score(test_chunks_df["label"], chunk_predictions)

print("Chunk-level Accuracy:", round(chunk_accuracy * 100, 2), "%")
print("\nChunk-level Classification Report:")
print(classification_report(
    test_chunks_df["label"],
    chunk_predictions,
    labels=[0, 1],
    target_names=["Anton Chekhov", "Guy de Maupassant"]
))

Chunk-level Accuracy: 95.41 %

Chunk-level Classification Report:
                   precision    recall  f1-score   support

    Anton Chekhov       1.00      0.90      0.95        48
Guy de Maupassant       0.92      1.00      0.96        61

         accuracy                           0.95       109
        macro avg       0.96      0.95      0.95       109
     weighted avg       0.96      0.95      0.95       109



# Further Ideas for exploration after the hackathon:

**Statistical analysis** of text using NLP, by analysis meaning of sentences, feature based grammars and analyzing structure of sentences!

reference: www.nltk.org/book